# 03 Feature Extraction

Extract feature vectors from YOLOv8 Pose keypoints.


In [ ]:
from pathlib import Path
import sys

import cv2

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from zola_robot.features.feature_extractor import FeatureExtractor
from zola_robot.io.video_sources import open_video_capture
from zola_robot.pose.pose_runner import PoseRunner


In [ ]:
SOURCE = 'camera'
MODEL_PATH = 'models/yolov8/yolov8n-pose.pt'


In [ ]:
extractor = FeatureExtractor()
cap = open_video_capture(SOURCE)

with PoseRunner(model_path=MODEL_PATH) as runner:
    while True:
        ret, frame = cap.read()
        if not ret:
            print('No frame received. Check source.')
            break

        result = runner.process_frame(frame)
        if result.landmarks is not None:
            height, width = frame.shape[:2]
            landmarks = result.landmarks.copy()
            landmarks[:, 0] /= width
            landmarks[:, 1] /= height
            features = extractor.extract(landmarks)
            print('Feature vector shape:', features.shape)

        cv2.imshow('Feature Extraction', result.annotated_frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()
